In [ ]:
#hide
! [ -e /content ] && pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

In [ ]:
# hide
from fastai.vision.all import *
from fastbook import *

matplotlib.rc("image", cmap="Greys")

# MNIST: All 10 Digits

This notebook follows [Chapter 4](../04_mnist_basics.ipynb), extending the 3-vs-7 classifier to all ten digits using the **full** MNIST dataset (`URLs.MNIST`).

## Load the full MNIST dataset

The sample dataset in the chapter only has folders for 3 and 7. Full MNIST uses `training` and `testing` splits with one subfolder per digit (0–9).

In [ ]:
path = untar_data(URLs.MNIST)
Path.BASE_PATH = path
path.ls()

In [ ]:
(path / "training").ls()

In [ ]:
def digit_files(split, digit):
    return (path / split / str(digit)).ls().sorted()


show_image(Image.open(digit_files("training", 8)[0]))

## Stack images into tensors

Same approach as the chapter: open each PNG, stack into a rank-3 tensor, scale pixels to 0–1.

In [ ]:
def load_digit_stack(split, digit):
    tensors = [tensor(Image.open(o)) for o in digit_files(split, digit)]
    return torch.stack(tensors).float() / 255


train_stacked = [load_digit_stack("training", d) for d in range(10)]
valid_stacked = [load_digit_stack("testing", d) for d in range(10)]
[t.shape for t in train_stacked]

## First try: pixel similarity (10 classes)

For each digit we compute the mean image on the training set, then classify by nearest mean (same `mnist_distance` as the chapter).

In [ ]:
def mnist_distance(a, b):
    return (a - b).abs().mean((-1, -2))


train_means = torch.stack([t.mean(0) for t in train_stacked])
show_image(train_means[3])

In [ ]:
def distances_to_means(imgs, means):
    return (imgs.unsqueeze(1) - means.unsqueeze(0)).abs().mean((-1, -2))


def predict_nearest_mean(imgs, means):
    return distances_to_means(imgs, means).argmin(dim=1)


def pixel_baseline_accuracy(stacked_list, means):
    accs = []
    for digit, tens in enumerate(stacked_list):
        preds = predict_nearest_mean(tens, means)
        accs.append((preds == digit).float().mean())
    # return torch.stack(accs).mean().item()
    return [x.item() for x in accs]


digit_accuracies = pixel_baseline_accuracy(valid_stacked, train_means)
print("Digit accuracies:")
for k in range(len(digit_accuracies)):
    print(f"  {k}: {digit_accuracies[k]}")
print("Overall accuracy: ", sum(digit_accuracies) / len(digit_accuracies))

## Neural network from scratch

Build `train_x` / `train_y` for all digits. Labels are class indices 0–9 (for `cross_entropy`, not binary 0/1).

In [ ]:
def stack_xy(stacked_list):
    x = torch.cat(stacked_list).view(-1, 28 * 28)
    y = torch.cat([tensor([d] * len(t)) for d, t in enumerate(stacked_list)]).long()
    return x, y


train_x, train_y = stack_xy(train_stacked)
valid_x, valid_y = stack_xy(valid_stacked)
train_x.shape, train_y.shape

In [ ]:
from fastai.data.all import L, Datasets, DataLoader

# Create fastai Datasets from the zipped lists of tensors
train_ds = L(list(zip(train_x, train_y)))
valid_ds = L(list(zip(valid_x, valid_y)))

# Create fastai DataLoader objects
# These fastai DataLoaders will correctly have the `decode_batch` method.
dl = DataLoader(train_ds, bs=256, shuffle=True)
valid_dl = DataLoader(valid_ds, bs=256)

For multi-class problems we use `cross_entropy` (logits in, class index labels). Accuracy: `argmax` over the 10 outputs.

In [ ]:
#def mnist_loss(preds, targets):
    #return F.cross_entropy(preds, targets)

mnist_loss = CrossEntropyLossFlat()

def batch_accuracy(preds, targets):
    return (preds.argmax(dim=1) == targets).float().mean()


def validate_epoch(model):
    accs = [batch_accuracy(model(xb), yb) for xb, yb in valid_dl]
    return round(torch.stack(accs).mean().item(), 4)

In [ ]:
def calc_grad(xb, yb, model):
    loss = mnist_loss(model(xb), yb)
    loss.backward()


lr = 1.0
linear_model = nn.Linear(28 * 28, 10)
opt = SGD(linear_model.parameters(), lr)


def train_epoch(model):
    for xb, yb in dl:
        calc_grad(xb, yb, model)
        opt.step()
        opt.zero_grad()


def train_model(model, epochs):
    for _ in range(epochs):
        train_epoch(model)
        print(validate_epoch(model), end=" ")

In [ ]:
train_model(linear_model, 20)

## `Learner`: linear and simple net

Same `Learner` API as the chapter, with 10 outputs and `cross_entropy`.

In [ ]:
dls_linear_learn = DataLoaders(dl, valid_dl)
linear_learn = Learner(
    dls_linear_learn, linear_model,
    loss_func=mnist_loss,
    opt_func=SGD,  metrics=batch_accuracy
)
linear_learn.fit(10, lr=lr)

In [ ]:
simple_net = nn.Sequential(
    nn.Linear(28 * 28, 30),
    nn.ReLU(),
    nn.Linear(30, 10),
)
simple_net_learn = Learner(
    dls_linear_learn, simple_net,
    loss_func=mnist_loss,
    opt_func=SGD, metrics=batch_accuracy
)
simple_net_learn.fit(10, lr=0.1)

In [ ]:
from fastai.torch_core import TensorBase
import random

# Now that dls_linear_learn is correctly configured, Learner.predict should work.
# The Learner.predict method is encountering an AttributeError due to how the DataLoaders
# were constructed (directly from zipped tensors) and its internal decoding process.
# To get the prediction, we can directly call the model's forward pass.

test_idx = random.randint(0, len(valid_x) - 1)
x_in = TensorBase(valid_x[test_idx].unsqueeze(0))
y_in = valid_y[test_idx]
raw_output = simple_net_learn.model(x_in)
predicted_class = raw_output.argmax(dim=1).item()

print(f"Predicted class index: {predicted_class}, ground truth class index: {y_in}")

## Going deeper: `vision_learner`

As in the end of Chapter 4, a small CNN (ResNet-18) on image batches reaches much higher accuracy.


In [ ]:
dls_vision_learn  = ImageDataLoaders.from_folder(
    path, train='training', valid='testing',
    batch_tfms=Normalize(),
)
dls_vision_learn.show_batch(max_n=9, figsize=(4, 4))

In [ ]:
vision_learn = vision_learner(
    dls_vision_learn, resnet18, pretrained=False, metrics=accuracy,
)
vision_learn.fit_one_cycle(1, 0.1)

In [ ]:
image_file = digit_files("training", 0)[0]
img = PILImage.create(image_file)
img.show()
vision_learn.predict(img)

In [ ]:
interp = ClassificationInterpretation.from_learner(vision_learn)
interp.plot_confusion_matrix(figsize=(6, 6))